# Thermal expansion workflow with pheasy and machine-learned potentials

This notebook computes the thermal expansion with the `CTEMaker` workflow. The
forces come from three MACE potentials, and the Born charges and the
dielectric tensor for the non-analytical correction come from MACE-Field. No
DFT is needed.

## How to use this notebook

| Part | What it does | Needs | Time |
|---|---|---|---|
| A. Run the workflows yourself | Runs MgO step by step, then five materials with three potentials, then a convergence scan for Si | MACE-Field, the MACE model files and a Materials Project API key | about 11 minutes for MgO on 32 CPU cores, many hours for all runs |
| B. Our results | Draws every figure and table from the numbers of our runs | atomate2 only | seconds on a laptop |

**Only want the results? Run the cells of Part B.** Part B uses nothing from
Part A. In Jupyter, click the first code cell of Part B and choose "Run
Selected Cell and All Below". Do not use "Run All", since the cells of Part A
start the workflow runs.

Contents:

- Background
- Installation
- Part A. Run the workflows yourself
  - A1. MgO step by step
  - A2. Five materials and three potentials
  - A3. Converging the cutoff and the supercell for Si
  - A4. Saving the numbers for Part B
- Part B. Our results
  - B1. Thermal expansion of the five materials
  - B2. Elastic constants and Grüneisen parameters
  - B3. Default settings against finite displacements
  - B4. Si: the third-order cutoff
  - B5. Si: finite displacements as a reference
- Known limitations

## Background

The workflow computes the thermal expansion tensor from mode Grüneisen tensors.
The thermal stress of each phonon mode is its heat capacity times its Grüneisen
tensor. The strain that relaxes the summed thermal stress is the thermal
expansion:

$$\alpha = \frac{S}{N_q V} \sum_{q\nu} c_{q\nu} \gamma_{q\nu}$$

Here $S$ is the elastic compliance, $c_{q\nu}$ and $\gamma_{q\nu}$ are the heat
capacity and Grüneisen tensor of each mode, $N_q$ is the number of q-points and
$V$ is the cell volume.

The flow has four steps:

1. The structure is converted to the standard primitive cell and relaxed tightly.
2. The pheasy phonon workflow fits the second- and third-order force constants
   with LASSO. It uses randomly displaced supercells with 0.03 Å displacements.
3. The elastic workflow fits the elastic tensor on the same relaxed structure.
4. phono3py computes the mode Grüneisen tensors on a 12x12x12 q-point mesh, with
   the non-analytical correction from the Born charges and the dielectric
   tensor. The thermal expansion follows from the formula above.

Steps 2 and 3 do not depend on each other, so a workflow manager can run them
at the same time. The same workflow runs with VASP as
`atomate2.vasp.flows.cte.CTEMaker`.

## Installation

The `pheasy`, `phono3py` and `ase` extras are needed, plus MACE-Field and the
Materials Project client:

```
pip install 'atomate2[pheasy,phono3py,ase]' mp-api
pip install git+https://github.com/mdi-group/mace-field.git@45d5c5fa7b40a155855b3d155df1760e36849e64
```

MACE-Field is a fork of MACE that installs as `mace-torch`, so it replaces MACE.
It also runs the three MACE potentials below.

The `pheasy` extra pulls in pheasy and ALM, and the `phono3py` extra pulls in
phono3py. ALM is compiled from source. The hiPhive tutorial and the atomate2
VASP documentation list what the build needs.

Part B needs only atomate2 itself.

## Part A. Run the workflows yourself

The cells of Part A run the workflows. They need the installation above, the
model files below and a Materials Project API key. Part B shows our results
without running anything.

### The potentials

The notebook uses three MACE foundation models. All three are released under the
Academic Software License.

- **MACE-OMAT-0-medium.** `mace_mp(model="medium-omat-0")` downloads it once and
  caches it under `~/.cache/mace`.
- **MACE-MATPES-PBE-0** and **MACE-MATPES-r2SCAN-0.** These are MACE-OMAT-0
  fine-tuned on the MatPES data set at the PBE and r2SCAN levels. They need
  `mace-torch>=0.3.10`. Download the two files from the `mace_matpes_0` release of
  [mace-foundations](https://github.com/ACEsuit/mace-foundations/releases/tag/mace_matpes_0):
  - [MACE-matpes-pbe-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-pbe-omat-ft.model)
  - [MACE-matpes-r2scan-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-r2scan-omat-ft.model)

The Born charges and the high-frequency dielectric tensor come from MACE-Field.
Download `MACEField-MH-0-omat-dielectric.model` from the
[MACE-Field releases](https://github.com/mdi-group/mace-field/releases). The
force field documentation compares it with DFT in its notes on MACE-Field.

Set the paths to the three downloaded files in the next cell. The MgO example
below only needs MACE-OMAT-0-medium. Change `device` to `"cuda"` to run MACE on a
GPU.

`float64` is worth the cost here. The third-order force constants come from
small force differences between displaced supercells, where `float32` noise is
not negligible.

In [ ]:
import os
import warnings

# macOS: conda's llvm-openmp and torch's bundled libomp both load and the
# duplicate aborts the process. This must be set before torch is imported.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
warnings.filterwarnings("ignore")

from mace.calculators import mace_mp  # noqa: E402

# Replace the three paths with the files you downloaded.
MACE_FIELD_MODEL = "/path/to/MACEField-MH-0-omat-dielectric.model"
MODEL_FILES = {
    "OMAT-0-medium": "medium-omat-0",
    "MATPES-PBE-0": "/path/to/MACE-matpes-pbe-omat-ft.model",
    "MATPES-r2SCAN-0": "/path/to/MACE-matpes-r2scan-omat-ft.model",
}


def calc_kwargs(model: str) -> dict:
    """Return the calculator settings for one of the models above."""
    return {"model": MODEL_FILES[model], "device": "cpu", "default_dtype": "float64"}


_ = mace_mp(**calc_kwargs("OMAT-0-medium"))  # downloads and caches on first use

### One calculator for all jobs

Every force field job builds its own calculator. `run_locally` runs all of
them in this one Python process, and the workflow has a few hundred jobs.
Loading MACE for each of them grows the memory by about 150 MB per job. This
cell makes the jobs share one calculator. It is only needed when many jobs run
in one process, not with a workflow manager.

In [ ]:
from ase.calculators.calculator import Calculator

import atomate2.forcefields.utils as ff_utils

_ase_calculator = ff_utils.ase_calculator
_calculators: dict[str, Calculator] = {}


def shared_ase_calculator(calculator_meta: object, **kwargs: object) -> Calculator:
    """Build each calculator once and reuse it for every job."""
    key = repr((calculator_meta, sorted(kwargs.items())))
    if key not in _calculators:
        _calculators[key] = _ase_calculator(calculator_meta, **kwargs)
    _calculators[key].reset()
    return _calculators[key]


ff_utils.ase_calculator = shared_ase_calculator

### A1. MgO step by step

This part runs MgO with MACE-OMAT-0-medium. B1 in Part B shows our result.

#### The structure

The primitive cell of MgO (mp-1265) comes from the Materials Project. Set your
key first with `export MP_API_KEY=...`.

The Materials Project serves this cell rotated away from the cubic axes, and
the pheasy fits fail for such a cell. `CTEMaker` therefore converts the input to
the standard primitive cell before the relaxation. This is the default,
`use_symmetrized_structure="primitive"`.

In [ ]:
from mp_api.client import MPRester

if not os.environ.get("MP_API_KEY"):
    raise OSError("export MP_API_KEY before running this cell")

with MPRester() as mpr:
    structure = mpr.get_structure_by_material_id("mp-1265")
structure.lattice

#### Building the workflow

`CTEMaker.from_force_field_name` sets one force field for the relaxation, the
phonons and the elastic tensor. Any MACE model routes through `mace_mp`, and
the model name in `calculator_kwargs` selects the weights.

`born_maker` adds a MACE-Field calculation of the Born charges and the
high-frequency dielectric tensor of the relaxed structure. The phonon workflow
uses them for the non-analytical correction of the phonon bands and of the
mode Grüneisen tensors.

Everything else stays at the workflow defaults. The supercells have
`min_length=12` Å, which gives 128 atoms for MgO. The third-order force
constants use the one-shot fit, and the temperatures run from 0 K to 1000 K in
steps of 10 K.

In [ ]:
from atomate2.forcefields.flows.cte import CTEMaker
from atomate2.forcefields.jobs import ForceFieldDielectricMaker

maker = CTEMaker.from_force_field_name(
    "MACE-MP-0", calculator_kwargs=calc_kwargs("OMAT-0-medium")
)
maker.phonon_maker.born_maker = ForceFieldDielectricMaker(
    calculator_kwargs={"model": MACE_FIELD_MODEL}
)
flow = maker.make(structure)
flow.draw_graph().show()

#### Running the workflow

The flow has a little over 200 jobs, most of them force calculations on the
displaced supercells. On 32 CPU cores of a Perlmutter node the whole run took
about 11 minutes.

`create_folders=True` is needed, because the thermal expansion job reads the
force constants from the folder of the pheasy fit.

In [ ]:
from jobflow import JobStore, run_locally
from maggma.stores import MemoryStore

job_store = JobStore(MemoryStore(), additional_stores={"data": MemoryStore()})
responses = run_locally(flow, store=job_store, create_folders=True, ensure_success=True)
doc = responses[flow.output.uuid][1].output

#### Reading the output

The output is a `CTEDocument`. It holds the thermal expansion tensor at each
temperature for each fit method. The value at 0 K is zero, since the heat
capacity is zero there. MgO is cubic, so the three diagonal components are
equal.

The next cell prints the thermal expansion at 300 K and the mean Grüneisen
parameter. B1 plots the thermal expansion of our run against temperature.

In [ ]:
import numpy as np

(result,) = doc.results
temperatures = np.array(doc.temperatures)
alpha = np.array(result.thermal_expansion_tensor)
i300 = int(np.argmin(np.abs(temperatures - 300)))

{
    "alpha(300 K) in 1/K": float(alpha[i300, 0, 0]),
    "mean Grueneisen parameter at 300 K": float(
        np.trace(result.average_gruneisen[i300]) / 3
    ),
    "has imaginary modes": result.has_imaginary_modes,
}

#### Checking the fit

If the thermal expansion comes out as zero or very small, check the
third-order force constants first. All zeros means the LASSO fit dropped them.
The penalty chosen by cross validation should also lie inside the search
range, not on one of its bounds.

In [ ]:
import re
from pathlib import Path

import h5py

fit_dir = Path(doc.phonon_job_dir) / "one_shot"
log = (fit_dir / "pheasy_anharmonic_fit.log").read_text()
with h5py.File(fit_dir / "fc3.hdf5") as file:
    fc3 = file["fc3"][:]

{
    "LASSO penalty": re.findall(r"alpha_(?:min|max|opt):\s*\S+", log),
    "max |fc3| in eV/A^3": float(np.abs(fc3).max()),
}

### A2. Five materials and three potentials

The same workflow runs for NaCl, KCl, MgO, CaO and GaAs, and with all three
potentials. The function below builds and runs one workflow in its own folder.
A3 uses it as well.

The loop after it runs 15 workflows, so it is off by default. Set
`RUN_ALL = True` to run it. On 32 CPU cores of a Perlmutter node each run took
1 to 24 minutes.

In [ ]:
from pymatgen.core import Structure

from atomate2.common.schemas.cte import CTEDocument

MATERIALS = {
    "NaCl": "mp-22862",
    "KCl": "mp-23193",
    "MgO": "mp-1265",
    "CaO": "mp-2605",
    "GaAs": "mp-2534",
}


def run_cte(
    structure: Structure,
    model: str,
    root_dir: str,
    min_length: float | None = None,
    c3: float | None = None,
) -> CTEDocument:
    """Run the thermal expansion workflow and return the CTEDocument.

    min_length sets the supercell size in Angstrom. c3 sets the third-order
    cutoff in Bohr. None keeps the workflow default.
    """
    maker = CTEMaker.from_force_field_name(
        "MACE-MP-0", calculator_kwargs=calc_kwargs(model)
    )
    maker.phonon_maker.born_maker = ForceFieldDielectricMaker(
        calculator_kwargs={"model": MACE_FIELD_MODEL}
    )
    if min_length is not None:
        maker.phonon_maker.min_length = min_length
    if c3 is not None:
        maker.phonon_maker.fcs_cutoff_radius = [-1, c3, 10]
    flow = maker.make(structure)
    os.makedirs(root_dir, exist_ok=True)
    responses = run_locally(
        flow,
        store=JobStore(MemoryStore(), additional_stores={"data": MemoryStore()}),
        create_folders=True,
        root_dir=root_dir,
        ensure_success=True,
    )
    return responses[flow.output.uuid][1].output


def alpha_300k(doc: CTEDocument) -> float:
    """Linear thermal expansion at 300 K in 1/K, from the xx component."""
    (result,) = doc.results
    i300 = int(np.argmin(np.abs(np.array(doc.temperatures) - 300)))
    return float(np.array(result.thermal_expansion_tensor)[i300, 0, 0])

In [ ]:
RUN_ALL = False
if RUN_ALL:
    with MPRester() as mpr:
        structures = {
            name: mpr.get_structure_by_material_id(mp_id)
            for name, mp_id in MATERIALS.items()
        }
    docs_all = {
        (name, model): run_cte(structure, model, f"runs/{name}_{model}")
        for name, structure in structures.items()
        for model in MODEL_FILES
    }
    alpha_all = {key: alpha_300k(cte_doc) * 1e6 for key, cte_doc in docs_all.items()}
    print(alpha_all)  # noqa: T201

For each material with MACE-OMAT-0-medium, the next cell plots four panels:

- the phonon bands from the second-order force constants of the fit, with the
  non-analytical correction,
- the elastic constants C11, C12 and C44,
- the mode Grüneisen parameter of each mode on the q-point mesh against its
  frequency, as a third of the trace of its Grüneisen tensor,
- the linear thermal expansion.

The title of the third panel gives the mean Grüneisen parameter at 300 K. Each
mode is weighted with its heat capacity.

The plot needs the files of a run, so it only works after the loop above.
B2 in Part B gives the elastic constants and the mean Grüneisen parameters of
our runs as a table.

In [ ]:
import matplotlib.pyplot as plt
import phonopy
from phono3py.file_IO import read_fc2_from_hdf5


def plot_summary(doc: CTEDocument, root_dir: str, title: str) -> None:
    """Plot the bands, elastic constants, Grueneisen parameters and expansion."""
    phonon_dir = Path(doc.phonon_job_dir)
    # unit cell, supercell and Born charges of the run, force constants of the fit
    phonon = phonopy.load(phonon_dir / "phonopy.yaml", produce_fc=False, log_level=0)
    phonon.force_constants = read_fc2_from_hdf5(phonon_dir / "one_shot" / "fc2.hdf5")
    phonon.auto_band_structure()
    bands = phonon.band_structure
    with h5py.File(next(Path(root_dir).glob("*/gruneisen_one_shot.hdf5"))) as file:
        frequencies = file["frequency"][:]
        gruneisen = np.trace(file["gruneisen_tensor"][:], axis1=2, axis2=3) / 3
    (result,) = doc.results
    temperatures = np.array(doc.temperatures)
    i300 = int(np.argmin(np.abs(temperatures - 300)))
    elastic = np.array(doc.elastic_tensor)

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    fig.suptitle(title)
    ax = axes[0, 0]
    for distances, band_frequencies in zip(
        bands.distances, bands.frequencies, strict=True
    ):
        ax.plot(distances, band_frequencies, color="C0")
    labels = iter(bands.labels)
    ticks, names = [0.0], [next(labels)]
    for distances, connected in zip(
        bands.distances, bands.path_connections, strict=True
    ):
        ticks.append(distances[-1])
        label = next(labels)
        if not connected and (following := next(labels, None)):
            label = f"{label}|{following}"
        names.append(label)
    for tick in ticks:
        ax.axvline(tick, color="gray", lw=0.5)
    ax.set_xticks(ticks, names)
    ax.set_xlim(0, ticks[-1])
    ax.set_ylabel("Frequency (THz)")

    ax = axes[0, 1]
    bars = ax.bar(
        ["C$_{11}$", "C$_{12}$", "C$_{44}$"],
        [elastic[0, 0], elastic[0, 1], elastic[3, 3]],
    )
    ax.bar_label(bars, fmt="%.0f")
    ax.set_ylabel("Elastic constant (GPa)")

    ax = axes[1, 0]
    kept = frequencies > doc.min_frequency
    mean_300k = np.trace(result.average_gruneisen[i300]) / 3
    ax.scatter(frequencies[kept], gruneisen[kept], s=2)
    ax.axhline(0, color="gray", ls=":")
    ax.set_title(f"mean at 300 K: {mean_300k:.2f}")
    ax.set_xlabel("Frequency (THz)")
    ax.set_ylabel("Mode Grüneisen parameter")

    ax = axes[1, 1]
    alpha = np.array(result.thermal_expansion_tensor)
    ax.plot(temperatures, alpha[:, 0, 0] * 1e6)
    ax.set_xlabel("Temperature (K)")
    ax.set_ylabel("Linear thermal expansion (10$^{-6}$ K$^{-1}$)")
    fig.tight_layout()
    plt.show()


if RUN_ALL:
    for name in MATERIALS:
        plot_summary(
            docs_all[name, "OMAT-0-medium"], f"runs/{name}_OMAT-0-medium", name
        )

### A3. Converging the cutoff and the supercell for Si

In Si the transverse acoustic modes near the zone boundary have negative
Grüneisen parameters. Their contribution to the thermal stress partly cancels
that of the other modes. The thermal expansion is the small difference of two
larger terms. Small errors in the third-order force constants therefore change
it a lot. This is why the default settings are not enough for Si.

The two atoms of Si are related by inversion, so their Born charges are zero.
MACE-Field gives zero as well, so the non-analytical correction does not change
this section.

Two settings control the third-order force constants:

- `phonon_maker.min_length` sets the supercell. For the Si primitive cell, 12, 16
  and 21 Å give the 4x4x4, 5x5x5 and 6x6x6 supercells with 128, 250 and 432
  atoms. The default is 12 Å.
- `phonon_maker.fcs_cutoff_radius` sets the cutoff radius of each order in Bohr.
  The second entry is the third-order cutoff. The default is
  `[-1, 12, 10]`, so 12 Bohr.

A cutoff should stay below half the shortest distance between periodic images
of the supercell. Above it an atom starts to interact with its own images. For
Si this limit is 14.6, 18.3 and 21.9 Bohr for the 4x4x4, 5x5x5 and 6x6x6
supercells.

A larger cutoff adds third-order force constants. The workflow picks the number
of displaced supercells so that the fit has about 100 equations per free force
constant, up to 600 supercells. The cost therefore grows quickly with the
cutoff. The 6x6x6 run at 18 Bohr used 201 displaced supercells. At 20 Bohr it used 527,
and the LASSO fit needed more than the 220 GB of memory we gave it. We stopped
at 18 Bohr.

The scan below runs 11 settings for each potential, so it is off by default. B4 in Part B shows our results.

In [ ]:
SI_SCAN = {  # supercell: min_length in Angstrom and third-order cutoffs in Bohr
    "4x4x4": (12, (12, 14, 16)),
    "5x5x5": (16, (12, 14, 16, 18)),
    "6x6x6": (21, (12, 14, 16, 18)),
}

RUN_SI_SCAN = False
if RUN_SI_SCAN:
    with MPRester() as mpr:
        si = mpr.get_structure_by_material_id("mp-149")
    # alpha(300 K) in 1e-6/K, in the layout of ALPHA_SI in B4
    alpha_si = {
        model: {
            cell: {
                c3: round(
                    alpha_300k(
                        run_cte(
                            si,
                            model,
                            f"runs/Si_{model}_{min_length}A_{c3}bohr",
                            min_length,
                            c3,
                        )
                    )
                    * 1e6,
                    2,
                )
                for c3 in cutoffs
            }
            for cell, (min_length, cutoffs) in SI_SCAN.items()
        }
        for model in MODEL_FILES
    }
    print(alpha_si)  # noqa: T201

#### The finite-displacement references

B3 and B5 compare the fits with force constants from finite displacements. We
computed these with phono3py outside this notebook. There is no cutoff and no
fit. The thermal expansion uses the same Born charges. For Si we used
supercells from 4x4x4 to 10x10x10. The 10x10x10 supercell has 2000 atoms and
needed 6201 displaced supercells for each potential.

### A4. Saving the numbers for Part B

Part B reads the numbers of our 15 runs from
`tests/test_data/forcefields/cte_tutorial/cte_results.json.gz`. The next cell
writes this file from the runs of A2. To plot your own runs in Part B, write
them to another file and set `RESULTS_FILE` in Part B to it.

In [ ]:
from datetime import UTC, datetime
from importlib.metadata import version

from monty.serialization import dumpfn

RESULTS_FILE = "../tests/test_data/forcefields/cte_tutorial/cte_results.json.gz"


def export_results(docs_all: dict, path: str) -> None:
    """Write the numbers that Part B plots for each (material, potential) run."""
    runs = {}
    for (name, model), cte_doc in docs_all.items():
        (result,) = cte_doc.results
        temperatures = np.array(cte_doc.temperatures)
        i300 = int(np.argmin(np.abs(temperatures - 300)))
        alpha = np.array(result.thermal_expansion_tensor)[:, 0, 0] * 1e6
        elastic = np.array(cte_doc.elastic_tensor)
        runs.setdefault(name, {})[model] = {
            "alpha": np.round(alpha, 4).tolist(),
            "C11": round(float(elastic[0, 0]), 1),
            "C12": round(float(elastic[0, 1]), 1),
            "C44": round(float(elastic[3, 3]), 1),
            "mean_gruneisen_300K": round(
                float(np.trace(result.average_gruneisen[i300]) / 3), 3
            ),
            "supercell": "x".join(
                str(round(cte_doc.supercell_matrix[i][i])) for i in range(3)
            ),
            "has_imaginary_modes": result.has_imaginary_modes,
        }
    provenance = {
        "settings": "CTEMaker defaults, Born charges from MACE-Field, float64",
        "potentials": {model: Path(file).name for model, file in MODEL_FILES.items()},
        "born_charges": Path(MACE_FIELD_MODEL).name,
        "versions": {
            package: version(package)
            for package in ("atomate2", "mace-torch", "phonopy", "phono3py", "pheasy")
        },
        "date": datetime.now(UTC).date().isoformat(),
    }
    dumpfn(
        {"provenance": provenance, "temperatures": temperatures.tolist(), "runs": runs},
        path,
    )


if RUN_ALL:
    export_results(docs_all, RESULTS_FILE)

## Part B. Our results

**Start here if you only want the results.** The cells of Part B draw every
figure and table from the numbers of our runs. They use nothing from Part A,
need only atomate2 and run in seconds on a laptop.

The next cell loads the numbers of the 15 runs of A2 and prints where they
come from: the settings, the model files and the package versions.

In [ ]:
import matplotlib.pyplot as plt  # noqa: F811
import numpy as np
import pandas as pd
from monty.serialization import loadfn

RESULTS_FILE = "../tests/test_data/forcefields/cte_tutorial/cte_results.json.gz"
results = loadfn(RESULTS_FILE)
temperatures = np.array(results["temperatures"])
runs = results["runs"]
MATERIALS = list(runs)
MODELS = list(runs[MATERIALS[0]])
results["provenance"]

### B1. Thermal expansion of the five materials

The next cell plots the linear thermal expansion of the 15 runs and gives its
value at 300 K.

In [ ]:
i300 = int(np.argmin(np.abs(temperatures - 300)))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for ax, model in zip(axes, MODELS, strict=True):
    for name in MATERIALS:
        ax.plot(temperatures, runs[name][model]["alpha"], label=name)
    ax.set_title(model)
    ax.set_xlabel("Temperature (K)")
axes[0].set_ylabel("Linear thermal expansion (10$^{-6}$ K$^{-1}$)")
axes[2].legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()

alpha_300k = pd.DataFrame(
    {
        model: {name: runs[name][model]["alpha"][i300] for name in MATERIALS}
        for model in MODELS
    }
)
alpha_300k.round(1)

| Material | OMAT-0-medium | MATPES-PBE-0 | MATPES-r2SCAN-0 |
|---|---|---|---|
| NaCl | 39.4 | 44.4 | 32.7 |
| KCl | 41.4 | 47.0 | 18.7 |
| MgO | 11.1 | 31.4 | 11.3 |
| CaO | 13.8 | 14.3 | 11.0 |
| GaAs | 6.0 | 4.5 | 2.5 |

The table gives $\alpha$(300 K) in 1e-6 K⁻¹, as printed by the cell above.
With MACE-OMAT-0-medium, MgO gives 11.1e-6 K⁻¹. Experiment gives about
1.0e-5 K⁻¹ for MgO at room temperature.

### B2. Elastic constants and Grüneisen parameters

The bulk modulus of a cubic crystal is (C11 + 2 C12)/3. The mean Grüneisen
parameter at 300 K weights each mode with its heat capacity. The supercell is
the one the workflow chose with the default `min_length`.

In [ ]:
summary = pd.DataFrame(
    {
        (name, model): {
            "C11 (GPa)": run["C11"],
            "C12 (GPa)": run["C12"],
            "C44 (GPa)": run["C44"],
            "bulk modulus (GPa)": round((run["C11"] + 2 * run["C12"]) / 3, 1),
            "mean Grueneisen at 300 K": run["mean_gruneisen_300K"],
            "supercell": run["supercell"],
        }
        for name in MATERIALS
        for model in MODELS
        for run in [runs[name][model]]
    }
).T
summary

| Material | Potential | C11 | C12 | C44 | Bulk modulus | Mean Grüneisen at 300 K | Supercell |
|---|---|---|---|---|---|---|---|
| NaCl | OMAT-0-medium | 48.5 | 12.2 | 11.9 | 24.3 | 1.65 | 3x3x3 |
| NaCl | MATPES-PBE-0 | 45.0 | 11.1 | 11.1 | 22.4 | 1.70 | 3x3x3 |
| NaCl | MATPES-r2SCAN-0 | 68.2 | 8.5 | 13.8 | 28.4 | 1.50 | 4x4x4 |
| KCl | OMAT-0-medium | 39.0 | 6.4 | 6.3 | 17.3 | 1.71 | 3x3x3 |
| KCl | MATPES-PBE-0 | 38.2 | 6.7 | 6.8 | 17.2 | 1.92 | 3x3x3 |
| KCl | MATPES-r2SCAN-0 | 46.3 | 3.9 | 6.4 | 18.0 | 0.77 | 3x3x3 |
| MgO | OMAT-0-medium | 279.6 | 91.3 | 139.6 | 154.1 | 1.58 | 4x4x4 |
| MgO | MATPES-PBE-0 | 179.7 | 60.2 | 60.6 | 100.0 | 2.39 | 5x5x5 |
| MgO | MATPES-r2SCAN-0 | 301.1 | 93.4 | 143.8 | 162.6 | 1.63 | 5x5x5 |
| CaO | OMAT-0-medium | 196.2 | 52.3 | 72.6 | 100.3 | 1.66 | 4x4x4 |
| CaO | MATPES-PBE-0 | 167.6 | 42.7 | 60.5 | 84.3 | 1.40 | 4x4x4 |
| CaO | MATPES-r2SCAN-0 | 221.0 | 55.0 | 75.0 | 110.3 | 1.43 | 4x4x4 |
| GaAs | OMAT-0-medium | 95.0 | 49.0 | 48.9 | 64.3 | 0.70 | 3x3x3 |
| GaAs | MATPES-PBE-0 | 100.9 | 47.0 | 54.0 | 65.0 | 0.54 | 3x3x3 |
| GaAs | MATPES-r2SCAN-0 | 114.8 | 52.5 | 67.4 | 73.3 | 0.33 | 3x3x3 |

The elastic constants and the bulk modulus are in GPa, as printed by the cell
above. The bands and the mode Grüneisen parameter of each mode need the files
of a run, so `plot_summary` in A2 plots them for your own runs.

### B3. Default settings against finite displacements

The next cell compares $\alpha$(300 K) of B1 with finite displacements in the
largest supercell we ran.

In [ ]:
# alpha(300 K) in 1e-6/K from finite displacements in the largest supercell
# we ran
ALPHA_FD_LARGE = {
    "OMAT-0-medium": {"NaCl": 39.3, "KCl": 41.8, "MgO": 11.0, "CaO": 14.0, "GaAs": 6.7},
    "MATPES-PBE-0": {"NaCl": 39.9, "KCl": 44.4, "MgO": 39.2, "CaO": 15.7, "GaAs": 6.7},
    "MATPES-r2SCAN-0": {
        "NaCl": 33.7,
        "KCl": 19.5,
        "MgO": 12.3,
        "CaO": 11.8,
        "GaAs": 4.1,
    },
}

alpha_fd = pd.DataFrame(ALPHA_FD_LARGE)
pd.concat(
    {
        "default settings": alpha_300k.round(1),
        "finite displacements": alpha_fd,
        "difference (%)": (100 * (alpha_300k / alpha_fd - 1)).round(0).astype(int),
    },
    axis=1,
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
x = np.arange(len(MATERIALS))
for ax, model in zip(axes, MODELS, strict=True):
    ax.bar(x - 0.2, alpha_300k[model], 0.4, label="default settings")
    ax.bar(x + 0.2, alpha_fd.loc[MATERIALS, model], 0.4, label="finite displacements")
    ax.set_xticks(x, MATERIALS)
    ax.set_title(model)
axes[0].set_ylabel("$\\alpha$(300 K) (10$^{-6}$ K$^{-1}$)")
axes[2].legend()
plt.show()

| Material | OMAT-0-medium | MATPES-PBE-0 | MATPES-r2SCAN-0 |
|---|---|---|---|
| NaCl | 39.4 / 39.3 | 44.4 / 39.9 | 32.7 / 33.7 |
| KCl | 41.4 / 41.8 | 47.0 / 44.4 | 18.7 / 19.5 |
| MgO | 11.1 / 11.0 | 31.4 / 39.2 | 11.3 / 12.3 |
| CaO | 13.8 / 14.0 | 14.3 / 15.7 | 11.0 / 11.8 |
| GaAs | 6.0 / 6.7 | 4.5 / 6.7 | 2.5 / 4.1 |

The table gives $\alpha$(300 K) in 1e-6 K⁻¹, with the default settings / from
finite displacements in the largest supercell.

The first value is what the workflow gives with the default settings.
To check it, we computed the force constants of each run by finite
displacements with phono3py, with no cutoff and no fit. Their thermal
expansion uses the same Born charges. In the supercell of
the run the fits agree with finite displacements within 6%. The one exception
is GaAs with MATPES-r2SCAN-0, at 10%. So the fit is right for the supercell it
uses.

We then repeated the finite displacements in larger supercells. The second value
is from the largest one, with 686 to 2662 atoms. Between
the two largest supercells each value changes by about 3% or less.

- With MACE-OMAT-0-medium the default settings are within 2% of the
  large-supercell value for the four rock-salt materials.
- With the two MatPES potentials the default settings are off by up to 11% for
  the rock-salt materials, and by 20% for MgO with MATPES-PBE-0.
- For GaAs the default settings are 11% to 38% too low with all three
  potentials. GaAs has the zinc-blende structure of Si. B4 and B5 show the
  same effect for Si in more detail.

The potentials differ much more than these errors. Two results stand out.

- KCl with MATPES-r2SCAN-0 gives 19.5e-6 K⁻¹ in the large supercell. This is
  less than half the value of the other two potentials.
- MgO with MATPES-PBE-0 gives 39.2e-6 K⁻¹. This is more than three times the
  value of the other two potentials and of experiment. MATPES-PBE-0 also gives
  a much softer MgO. Its bulk modulus in B2 is 100 GPa, against 154 GPa
  with MACE-OMAT-0-medium and 163 GPa with MATPES-r2SCAN-0. A softer lattice
  expands more.

### B4. Si: the third-order cutoff

A3 explains the scan. The next cell holds our results for Si and plots
$\alpha$(300 K) against the third-order cutoff.

In [ ]:
# alpha(300 K) in 1e-6/K from our runs, by potential, supercell and cutoff in Bohr
ALPHA_SI = {
    "OMAT-0-medium": {
        "4x4x4": {12: 1.42, 14: 1.90, 16: 1.87},
        "5x5x5": {12: 1.53, 14: 2.04, 16: 2.27, 18: 2.06},
        "6x6x6": {12: 1.64, 14: 1.99, 16: 2.30, 18: 2.07},
    },
    "MATPES-PBE-0": {
        "4x4x4": {12: 3.25, 14: 4.13, 16: 4.06},
        "5x5x5": {12: 3.35, 14: 4.30, 16: 3.91, 18: 3.53},
        "6x6x6": {12: 3.43, 14: 4.32, 16: 3.85, 18: 3.48},
    },
    "MATPES-r2SCAN-0": {
        "4x4x4": {12: -0.83, 14: -0.46, 16: -0.49},
        "5x5x5": {12: -0.85, 14: -0.32, 16: -0.61, 18: 0.03},
        "6x6x6": {12: -0.52, 14: -0.37, 16: -0.64, 18: 0.08},
    },
}
IMAGE_LIMIT = {"4x4x4": 14.6, "5x5x5": 18.3, "6x6x6": 21.9}  # Bohr
# finite displacements, by potential and supercell n x n x n
ALPHA_FD = {
    "OMAT-0-medium": {4: 1.74, 5: 1.96, 6: 2.32, 7: 2.64, 8: 2.83, 9: 2.89, 10: 2.90},
    "MATPES-PBE-0": {4: 3.75, 5: 3.41, 6: 3.45, 7: 3.58, 8: 3.67, 9: 3.70, 10: 3.71},
    "MATPES-r2SCAN-0": {
        4: -0.53,
        5: -0.10,
        6: 0.14,
        7: 0.44,
        8: 0.56,
        9: 0.57,
        10: 0.58,
    },
}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for ax, (model, by_cell) in zip(axes, ALPHA_SI.items(), strict=True):
    for cell, by_cutoff in by_cell.items():
        cutoffs = np.array(list(by_cutoff))
        values = np.array(list(by_cutoff.values()))
        (line,) = ax.plot(cutoffs, values, "o-", label=cell)
        beyond = cutoffs > IMAGE_LIMIT[cell]
        ax.plot(
            cutoffs[beyond], values[beyond], "o", color=line.get_color(), mfc="white"
        )
    ax.axhline(ALPHA_FD[model][10], color="black", ls=":", label="finite displacements")
    ax.axhline(2.6, color="gray", ls="--", label="experiment")
    ax.set_title(model)
    ax.set_xlabel("Third-order cutoff (Bohr)")
axes[0].set_ylabel("$\\alpha$(300 K) (10$^{-6}$ K$^{-1}$)")
axes[0].legend()
plt.show()

| Supercell | Cutoff (Bohr) | OMAT-0-medium | MATPES-PBE-0 | MATPES-r2SCAN-0 |
|---|---|---|---|---|
| 4x4x4 | 12 | 1.42 | 3.25 | -0.83 |
| 4x4x4 | 14 | 1.90 | 4.13 | -0.46 |
| 4x4x4 | 16 | 1.87 | 4.06 | -0.49 |
| 5x5x5 | 12 | 1.53 | 3.35 | -0.85 |
| 5x5x5 | 14 | 2.04 | 4.30 | -0.32 |
| 5x5x5 | 16 | 2.27 | 3.91 | -0.61 |
| 5x5x5 | 18 | 2.06 | 3.53 | 0.03 |
| 6x6x6 | 12 | 1.64 | 3.43 | -0.52 |
| 6x6x6 | 14 | 1.99 | 4.32 | -0.37 |
| 6x6x6 | 16 | 2.30 | 3.85 | -0.64 |
| 6x6x6 | 18 | 2.07 | 3.48 | 0.08 |

The table gives $\alpha$(300 K) of Si in 1e-6 K⁻¹ from the fits.

Open markers are cutoffs above the image limit of the supercell. The dotted
line is the finite-displacement value in the 10x10x10 supercell from B5. The
dashed line is the experimental value of about 2.6e-6 K⁻¹ (Okada and Tokumaru,
J. Appl. Phys. 56, 314 (1984)).

- The default 12 Bohr cutoff is far from the converged value. With
  MACE-OMAT-0-medium it gives about half of it. With MATPES-r2SCAN-0 even the
  sign is wrong.
- Between 12 and 18 Bohr the value goes up and down as each new shell of
  neighbors enters the fit. It does not settle within the cutoffs we could
  afford.
- From 14 Bohr on, the value at a fixed cutoff changes little between the 5x5x5
  and 6x6x6 supercells. The cutoff limits the result, more than the supercell.

### B5. Si: finite displacements as a reference

The next cell plots $\alpha$(300 K) of Si from finite displacements against
the supercell size.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.6))
for model, by_size in ALPHA_FD.items():
    ax.plot(list(by_size), list(by_size.values()), "o-", label=model)
ax.axhline(2.6, color="gray", ls="--", label="experiment")
ax.set_xlabel("Supercell size n (n x n x n)")
ax.set_ylabel("$\\alpha$(300 K) (10$^{-6}$ K$^{-1}$)")
ax.legend()
plt.show()

| Supercell | OMAT-0-medium | MATPES-PBE-0 | MATPES-r2SCAN-0 |
|---|---|---|---|
| 4x4x4 | 1.74 | 3.75 | -0.53 |
| 5x5x5 | 1.96 | 3.41 | -0.10 |
| 6x6x6 | 2.32 | 3.45 | 0.14 |
| 7x7x7 | 2.64 | 3.58 | 0.44 |
| 8x8x8 | 2.83 | 3.67 | 0.56 |
| 9x9x9 | 2.89 | 3.70 | 0.57 |
| 10x10x10 | 2.90 | 3.71 | 0.58 |

The table gives $\alpha$(300 K) of Si in 1e-6 K⁻¹ from finite displacements.

- From 9x9x9 to 10x10x10 the value changes by less than 0.02e-6 K⁻¹. It is
  converged at 2.90e-6 K⁻¹ with MACE-OMAT-0-medium, 3.71e-6 K⁻¹ with
  MATPES-PBE-0 and 0.58e-6 K⁻¹ with MATPES-r2SCAN-0.
- We also cut the 8x8x8 finite-displacement force constants at the pheasy
  cutoffs and recomputed the thermal expansion. At 16 and 18 Bohr this agrees
  with the 6x6x6 fits within 0.05e-6 K⁻¹. The fits are right for their cutoff.
- The value only settles in the 9x9x9 and 10x10x10 supercells. They hold
  interactions up to 17 and 19 Å. Each of these MACE potentials sees the
  neighbors within 12 Å of an atom, through two layers with a 6 Å cutoff. So
  the force constants can couple atoms up to 24 Å apart.
- A pheasy fit with a cutoff of about 19 Å, or 36 Bohr, is far beyond what the
  workflow can do. The fit at 20 Bohr already ran out of memory.
- MACE-OMAT-0-medium gives 12% more than experiment and MATPES-PBE-0 gives 43%
  more. MATPES-r2SCAN-0 gives about a fifth of the experimental value.

Si is a hard case, because its thermal expansion is a small difference of two
larger terms. For such a material, compare the result with finite displacements
in growing supercells. In B3 the default supercell is within 2% for the
rock-salt materials with MACE-OMAT-0-medium. With the other potentials, and for
GaAs, a larger supercell changes the result by up to 38%.

## Known limitations

The frequencies and Grüneisen tensors are those of the relaxed structure. They
are not renormalized with temperature, so the result is least reliable at high
temperature.

The Born charges and the dielectric tensor come from MACE-Field. On 50
materials that are likely not in its training data, its Born charges are off by
about 11% from DFT and its dielectric constants by about 8%. With VASP, the Born
charges of the phonon run are used.

If a frequency on the q-point mesh is below -0.1 THz, a warning is raised and
the thermal expansion of that fit is not computed.